In [ ]:
'''
Relacionar google drive con Collab, para poder usar mi drive como almacenamiento de archivos,
y como lugar donde van a reposar los outputs que se generen en este script.
'''

from google.colab import drive
drive.mount('/content/t4drive')
# Ejecutar el código anterior y aceptar lo solicitado; permisos y accesos

In [2]:
'''
Importación de paquetes módulos y funciones. Cabe aclarar que no es necesario
importarlastodos al inicio, solo que es una práctica heredada de la codificación
tradicional que ayuda a ser mas eficiente la interpretación del código así como
su lectura.
'''

import os # Para interactuar con el sistema operativo (rutas de archivos, crear directorios)
import pandas as pd # Para manipulación y análisis de datos (DataFrames)
import matplotlib.pyplot as plt # Para crear visualizaciones estáticas (gráficos)
import seaborn as sns # Para crear visualizaciones estadísticas atractivas, basado en matplotlib
from sklearn.model_selection import train_test_split # Para dividir datos en conjuntos de entrenamiento, desarrollo y prueba
from sklearn.linear_model import LogisticRegression # Implementación del modelo de Regresión Logística
from sklearn.metrics import accuracy_score, classification_report # Métricas para evaluar modelos de clasificación (precisión, informe de clasificación)
from sklearn.tree import DecisionTreeClassifier # Implementación del modelo de Árbol de Decisión
from sklearn.model_selection import GridSearchCV # Para ajuste de hiperparámetros utilizando búsqueda en cuadrícula y validación cruzada
from sklearn.neighbors import KNeighborsClassifier # Implementación del modelo k-NN (K-Nearest Neighbors)
from google.colab import files # Para interactuar con archivos en Google Colab (subir, descargar)
from sklearn.datasets import make_blobs, make_moons # Para generar datasets sintéticos tipo 'Blob'
import numpy as np # Para operaciones numéricas y manipulación de arrays
from sklearn.svm import SVC # Implementación del modelo de Máquinas de Soporte Vectorial (SVM)
from sklearn.neural_network import MLPClassifier # Implementación del modelo de Perceptrón Multicapa (MLP)
from matplotlib.colors import ListedColormap # Para crear mapas de colores personalizados en visualizaciones
import time # Para medir el tiempo de ejecución
from sklearn.datasets import make_moons # Para generar datasets sintéticos tipo 'Moons'
from sklearn.metrics import classification_report, accuracy_score
import warnings
from sklearn.exceptions import ConvergenceWarning

In [11]:
'''
Definir las rutas para guardar las figuras, tablas y notebooks.
'''

# Definir las rutas base
base = '/content/t4drive/MyDrive/taller4'
figs = os.path.join(base, 'figures')
codes = os.path.join(base, 'codes')

# Crear las rutas sino existen
os.makedirs(figs, exist_ok=True)
os.makedirs(codes, exist_ok=True)

Escenario 1: Binario vs. Clasificación multiclase

In [ ]:
'''
1. Generar los siguientes tres datasets (cada 1 con 1000 registros):

- Con dos clases
- Con cinco clases
- Con veinte clases

NOTA: Los datasets deben ser de tipo 'Blob'. Son datasets que se distribuyen como
puntos en un plano cartesiano, y cada clase agremia sus puntos en una región del
espacio cartesiano.
'''

# Función para crear un dataset desbalanceado con submuestreo aleatorio
def create_imbalanced_blob(n_samples, centers, random_state, total_samples_target=1000):
    # Generar datos balanceados iniciales (generar más muestras de las necesarias para permitir el submuestreo)
    # Heurística ajustada para asegurar suficientes muestras iniciales y al menos 5 muestras por clase objetivo para cv=5
    initial_samples = max(n_samples * 2, int(total_samples_target * 2), centers * 10) # Ensure enough initial samples for sampling and CV
    X, y = make_blobs(n_samples=initial_samples, centers=centers, random_state=random_state)
    df = pd.DataFrame(X, columns=['feature1', 'feature2'])
    df['class'] = y

    if centers > 1:
        # Determinar conteos de clases y ordenar para identificar fácilmente mayorías/minorías potenciales
        class_counts = df['class'].value_counts().sort_values(ascending=False)
        classes = class_counts.index.tolist()

        imbalanced_df_list = []
        current_total_samples = 0

        # Decidir el número objetivo de muestras para cada clase (aleatoriamente)
        # Asegurar al menos cinco muestras por clase si es posible for cv=5
        target_samples_per_class = {}
        random_proportions = np.random.rand(centers)
        random_proportions = random_proportions / random_proportions.sum() if random_proportions.sum() > 0 else np.ones(centers) / centers # Normalize proportions

        # Calculate target samples based on proportions, ensuring at least 5 samples if possible
        min_samples_per_class = 5 # Minimum samples required for cv=5
        for i, class_label in enumerate(classes):
            target_samples = max(min_samples_per_class, int(total_samples_target * random_proportions[i]))
            # Asegurar que las muestras objetivo no excedan las muestras disponibles
            available_samples = len(df[df['class'] == class_label])
            target_samples_per_class[class_label] = min(target_samples, available_samples)

        # Adjust target samples if the sum is not close to total_samples_target
        current_sum = sum(target_samples_per_class.values())
        if current_sum < total_samples_target:
             # Distribute the remaining samples, giving more to classes that were heavily undersampled
             remaining_to_distribute = total_samples_target - current_sum
             # Enfoque simple: añadir proporcionalmente a los objetivos existentes
             total_current_targets = sum(target_samples_per_class.values())
             if total_current_targets > 0:
                 for class_label in classes:
                     add_samples = int(remaining_to_distribute * (target_samples_per_class[class_label] / total_current_targets))
                     available_samples = len(df[df['class'] == class_label])
                     target_samples_per_class[class_label] = min(available_samples, target_samples_per_class[class_label] + add_samples)
             # Redistribute any remaining if sum still less than target
             current_sum = sum(target_samples_per_class.values())
             if current_sum < total_samples_target:
                 diff = total_samples_target - current_sum
                 for class_label in classes:
                     if diff > 0:
                         target_samples_per_class[class_label] += 1
                         diff -= 1
                     else:
                         break


        # Realizar el muestreo basado en las muestras objetivo ajustadas
        imbalanced_df_list = []
        for class_label in classes:
            current_class_df = df[df['class'] == class_label]
            if len(current_class_df) > 0:
                num_samples = target_samples_per_class.get(class_label, 0)
                if num_samples > 0:
                    replace = num_samples > len(current_class_df) # No debería ser necesario con la verificación min anterior, pero como salvaguarda
                    imbalanced_df_list.append(current_class_df.sample(n=num_samples, replace=replace, random_state=random_state))


        imbalanced_df = pd.concat(imbalanced_df_list).sample(frac=1, random_state=random_state).reset_index(drop=True)
    else:
        imbalanced_df = df.sample(n=total_samples_target, random_state=random_state).reset_index(drop=True)


    return imbalanced_df

# Generar dataset con 2 clases con desbalanceo aleatorio en la cantidad de registros (total ~1000)
df_2 = create_imbalanced_blob(n_samples=1000, centers=2, random_state=42, total_samples_target=1000)
print("Dataset con 2 clases (con desbalance aleatorio):")
display(df_2.head())
print("\nDistribución de clases en el dataset con 2 clases (desbalanceado):")
display(df_2['class'].value_counts())

# Generar dataset con 5 clases con desbalanceo aleatorio en la cantidad de registros (total ~1000)
df_5 = create_imbalanced_blob(n_samples=1000, centers=5, random_state=42, total_samples_target=1000)
print("\nDataset con 5 clases (con desbalance aleatorio):")
display(df_5.head())
print("\nDistribución de clases en el dataset con 5 clases (desbalanceado):")
display(df_5['class'].value_counts())

# Generar dataset con 20 clases con desbalanceo aleatorio en la cantidad de registros (total ~1000)
df_20 = create_imbalanced_blob(n_samples=1000, centers=20, random_state=42, total_samples_target=1000)
print("\nDataset con 20 clases (con desbalance aleatorio):")
display(df_20.head())
print("\nDistribución de clases en el dataset con 20 clases (desbalanceado):")
display(df_20['class'].value_counts().sort_index())

In [ ]:
'''
Generar un histograma para cada dataset que permita visualizar la frecuencia de cada
una de las clases.
'''

# Histograma para el dataset con 2 clases
plt.figure(figsize=(8, 6))
sns.countplot(x='class', data=df_2)
plt.title('Distribución de clases en el dataset con 2 clases')
plt.xlabel('Clase')
plt.ylabel('Frecuencia')
plt.savefig(os.path.join(figs, 'histogram_2_classes.png')) # Save the figure
plt.show()

# Histograma para el dataset con 5 clases
plt.figure(figsize=(8, 6))
sns.countplot(x='class', data=df_5)
plt.title('Distribución de clases en el dataset con 5 clases')
plt.xlabel('Clase')
plt.ylabel('Frecuencia')
plt.savefig(os.path.join(figs, 'histogram_5_classes.png')) # Save the figure
plt.show()

# Histograma para el dataset con 20 clases
plt.figure(figsize=(12, 6))
sns.countplot(x='class', data=df_20)
plt.title('Distribución de clases en el dataset con 20 clases')
plt.xlabel('Clase')
plt.ylabel('Frecuencia')
plt.xticks(rotation=90) # Rotar las etiquetas del eje x para mejor legibilidad con 20 clases
plt.savefig(os.path.join(figs, 'histogram_20_classes.png')) # Save the figure
plt.show()

In [ ]:
'''
Dividir cada dataset generado en conjuntos de entrenamiento, desarrollo y prueba.
NOTA: Cada nuevo dataset, mantuvo la proporción de clases del dataset padre, además,
se dividieron en una proporción de 60%, 20% y 20% respectivamente. Mantener la proporción
de clases en todos los datasets.
'''

# Dividir df_2
X_2 = df_2[['feature1', 'feature2']]
y_2 = df_2['class']
X_2_train, X_2_temp, y_2_train, y_2_temp = train_test_split(X_2, y_2, test_size=0.4, random_state=42, stratify=y_2) # 40% para temporal (desarrollo+prueba)
X_2_dev, X_2_test, y_2_dev, y_2_test = train_test_split(X_2_temp, y_2_temp, test_size=0.5, random_state=42, stratify=y_2_temp) # Dividir temporal 50/50 para desarrollo y prueba

print("Dataset con 2 clases:")
print(f"Training set shape: {X_2_train.shape}")
print(f"Development set shape: {X_2_dev.shape}")
print(f"Test set shape: {X_2_test.shape}")
print("\nDistribución de clases en y_2_train:")
display(y_2_train.value_counts(normalize=True))
print("Distribución de clases en y_2_dev:")
display(y_2_dev.value_counts(normalize=True))
print("Distribución de clases en y_2_test:")
display(y_2_test.value_counts(normalize=True))

# Dividir df_5
X_5 = df_5[['feature1', 'feature2']]
y_5 = df_5['class']
X_5_train, X_5_temp, y_5_train, y_5_temp = train_test_split(X_5, y_5, test_size=0.4, random_state=42, stratify=y_5)
X_5_dev, X_5_test, y_5_dev, y_5_test = train_test_split(X_5_temp, y_5_temp, test_size=0.5, random_state=42, stratify=y_5_temp)

print("\nDataset con 5 clases:")
print(f"Training set shape: {X_5_train.shape}")
print(f"Development set shape: {X_5_dev.shape}")
print(f"Test set shape: {X_5_test.shape}")
print("\nDistribución de clases en y_5_train:")
display(y_5_train.value_counts(normalize=True))
print("Distribución de clases en y_5_dev:")
display(y_5_dev.value_counts(normalize=True))
print("Distribución de clases en y_5_test:")
display(y_5_test.value_counts(normalize=True))

# Dividir df_20
X_20 = df_20[['feature1', 'feature2']]
y_20 = df_20['class']
X_20_train, X_20_temp, y_20_train, y_20_temp = train_test_split(X_20, y_20, test_size=0.4, random_state=42, stratify=y_20)
X_20_dev, X_20_test, y_20_dev, y_20_test = train_test_split(X_20_temp, y_20_temp, test_size=0.5, random_state=42, stratify=y_20_temp)

print("\nDataset con 20 clases:")
print(f"Training set shape: {X_20_train.shape}")
print(f"Development set shape: {X_20_dev.shape}")
print(f"Test set shape: {X_20_test.shape}")
print("\nDistribución de clases en y_20_train:")
display(y_20_train.value_counts(normalize=True))
print("Distribución de clases en y_20_dev:")
display(y_20_dev.value_counts(normalize=True))
print("Distribución de clases en y_20_test:")
display(y_20_test.value_counts(normalize=True))

In [ ]:
'''
Considerar clasificador:
  - K-NN
'''
# Diccionario para almacenar los mejores modelos y reportes de clasificación
knn_results = {}

# Definir el rango de vecinos a probar
k_range = range(1, 31)

# --- Dataset con 2 clases ---
print("--- Clasificador K-NN para 2 Clases ---")
X_train, y_train = X_2_train, y_2_train
X_dev, y_dev = X_2_dev, y_2_dev
X_test, y_test = X_2_test, y_2_test

best_k_2 = None
best_accuracy_2 = -1

# Paso 1 y 2: Entrenar en el conjunto de entrenamiento y encontrar el mejor n_neighbors usando el conjunto de desarrollo
print("Buscando el mejor n_neighbors usando el conjunto de desarrollo...")
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train) # Entrenar en el conjunto de entrenamiento
    y_dev_pred = knn.predict(X_dev) # Predecir en el conjunto de desarrollo
    accuracy = accuracy_score(y_dev, y_dev_pred) # Evaluar en el conjunto de desarrollo

    if accuracy > best_accuracy_2:
        best_accuracy_2 = accuracy
        best_k_2 = k

print(f"Mejor n_neighbors para 2 clases: {best_k_2}")

# Paso 3: Entrenar el modelo final en el conjunto de entrenamiento con el best k encontrado en el conjunto de desarrollo y evaluar en el conjunto de prueba
print(f"Entrenando el modelo final en el conjunto de entrenamiento con el mejor k={best_k_2} y evaluando en el conjunto de prueba...")
best_knn_2 = KNeighborsClassifier(n_neighbors=best_k_2)
best_knn_2.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_2 = best_knn_2.predict(X_test)
report_2 = classification_report(y_test, y_test_pred_2)
print("\nReporte de Clasificación (Conjunto de Prueba - 2 Clases):")
print(report_2)
knn_results['2_classes'] = {'best_k': best_k_2, 'report': report_2, 'model': best_knn_2}


# --- Dataset con 5 clases ---
print("\n--- Clasificador K-NN para 5 Clases ---")
X_train, y_train = X_5_train, y_5_train
X_dev, y_dev = X_5_dev, y_5_dev
X_test, y_test = X_5_test, y_5_test

best_k_5 = None
best_accuracy_5 = -1

# Paso 1 y 2: Entrenar en el conjunto de entrenamiento y encontrar el mejor n_neighbors usando el conjunto de desarrollo
print("Buscando el mejor n_neighbors usando el conjunto de desarrollo...")
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train) # Entrenar en el conjunto de entrenamiento
    y_dev_pred = knn.predict(X_dev) # Predecir en el conjunto de desarrollo
    accuracy = accuracy_score(y_dev, y_dev_pred) # Evaluar en el conjunto de desarrollo

    if accuracy > best_accuracy_5:
        best_accuracy_5 = accuracy
        best_k_5 = k

print(f"Mejor n_neighbors para 5 clases: {best_k_5}")

# Paso 3: Entrenar el modelo final en el conjunto de entrenamiento con el best k encontrado en el conjunto de desarrollo y evaluar en el conjunto de prueba
print(f"Entrenando el modelo final en el conjunto de entrenamiento con el mejor k={best_k_5} y evaluando en el conjunto de prueba...")
best_knn_5 = KNeighborsClassifier(n_neighbors=best_k_5)
best_knn_5.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_5 = best_knn_5.predict(X_test)
report_5 = classification_report(y_test, y_test_pred_5)
print("\nReporte de Clasificación (Conjunto de Prueba - 5 Clases):")
print(report_5)
knn_results['5_classes'] = {'best_k': best_k_5, 'report': report_5, 'model': best_knn_5}


# --- Dataset con 20 clases ---
print("\n--- Clasificador K-NN para 20 Clases ---")
X_train, y_train = X_20_train, y_20_train
X_dev, y_dev = X_20_dev, y_20_dev
X_test, y_test = X_20_test, y_20_test

best_k_20 = None
best_accuracy_20 = -1

# Paso 1 y 2: Entrenar en el conjunto de entrenamiento y encontrar el mejor n_neighbors usando el conjunto de desarrollo
print("Buscando el mejor n_neighbors usando el conjunto de desarrollo...")
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train) # Entrenar en el conjunto de entrenamiento
    y_dev_pred = knn.predict(X_dev) # Predecir en el conjunto de desarrollo
    accuracy = accuracy_score(y_dev, y_dev_pred) # Evaluar en el conjunto de desarrollo

    if accuracy > best_accuracy_20:
        best_accuracy_20 = accuracy
        best_k_20 = k

print(f"Mejor n_neighbors para 20 clases: {best_k_20}")

# Paso 3: Entrenar el modelo final en el conjunto de entrenamiento con el best k encontrado en el conjunto de desarrollo y evaluar en el conjunto de prueba
print(f"Entrenando el modelo final en el conjunto de entrenamiento con el mejor k={best_k_20} y evaluando en el conjunto de prueba...")
best_knn_20 = KNeighborsClassifier(n_neighbors=best_k_20)
best_knn_20.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_20 = best_knn_20.predict(X_test)
report_20 = classification_report(y_test, y_test_pred_20)
print("\nReporte de Clasificación (Conjunto de Prueba - 20 Clases):")
print(report_20)
knn_results['20_classes'] = {'best_k': best_k_20, 'report': report_20, 'model': best_knn_20}

print("\nResultados de K-NN almacenados en el diccionario 'knn_results'.")



In [ ]:
'''
Considerar clasificador:
  - Árbol de decisión
'''

# Diccionario para almacenar los mejores modelos y reportes de clasificación del Árbol de Decisión
dt_results = {}

# Definir el rango de hiperparámetros a probar (ejemplo basado en hiperparámetros comunes)
# Si la imagen adjunta especifica otros hiperparámetros o rangos, por favor, házmelo saber.
param_grid_dt = {
    'max_depth': [None, 5, 10, 15, 2], # Profundidad máxima del árbol
    'min_samples_split': [2, 5, 10], # Número mínimo de muestras requeridas para dividir un nodo interno
    'min_samples_leaf': [1, 2, 4] # Número mínimo de muestras requeridas para ser una hoja
}

# Función para encontrar los mejores hiperparámetros usando el conjunto de desarrollo
def find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid):
    best_score = -1
    best_params = None

    # Vamos a probar todas las combinaciones de hiperparámetros
    for max_depth in param_grid['max_depth']:
        for min_samples_split in param_grid['min_samples_split']:
            for min_samples_leaf in param_grid['min_samples_leaf']:
                # Crear el clasificador con los hiperparámetros actuales
                dt = DecisionTreeClassifier(
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    random_state=42 # Para reproducibilidad
                )

                # Entrenar en el conjunto de entrenamiento
                dt.fit(X_train, y_train)

                # Evaluar en el conjunto de desarrollo
                y_dev_pred = dt.predict(X_dev)
                accuracy = accuracy_score(y_dev, y_dev_pred)

                # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                if accuracy > best_score:
                    best_score = accuracy
                    best_params = {'max_depth': max_depth, 'min_samples_split': min_samples_split, 'min_samples_leaf': min_samples_leaf}

    return best_params

# --- Dataset con 2 clases ---
print("--- Clasificador Árbol de Decisión para 2 Clases ---")
X_train, y_train = X_2_train, y_2_train
X_dev, y_dev = X_2_dev, y_2_dev
X_test, y_test = X_2_test, y_2_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_dt_params_2 = find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid_dt)
print(f"Mejores hiperparámetros para 2 clases: {best_dt_params_2}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_dt_2 = DecisionTreeClassifier(**best_dt_params_2, random_state=42)
best_dt_2.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_2 = best_dt_2.predict(X_test)
report_2 = classification_report(y_test, y_test_pred_2)
print("\nReporte de Clasificación (Conjunto de Prueba - 2 Clases):")
print(report_2)
dt_results['2_classes'] = {'best_params': best_dt_params_2, 'report': report_2, 'model': best_dt_2}


# --- Dataset con 5 clases ---
print("\n--- Clasificador Árbol de Decisión para 5 Clases ---")
X_train, y_train = X_5_train, y_5_train
X_dev, y_dev = X_5_dev, y_5_dev
X_test, y_test = X_5_test, y_5_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_dt_params_5 = find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid_dt)
print(f"Mejores hiperparámetros para 5 clases: {best_dt_params_5}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_dt_5 = DecisionTreeClassifier(**best_dt_params_5, random_state=42)
best_dt_5.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_5 = best_dt_5.predict(X_test)
report_5 = classification_report(y_test, y_test_pred_5)
print("\nReporte de Clasificación (Conjunto de Prueba - 5 Clases):")
print(report_5)
dt_results['5_classes'] = {'best_params': best_dt_params_5, 'report': report_5, 'model': best_dt_5}


# --- Dataset con 20 clases ---
print("\n--- Clasificador Árbol de Decisión para 20 Clases ---")
X_train, y_train = X_20_train, y_20_train
X_dev, y_dev = X_20_dev, y_20_dev
X_test, y_test = X_20_test, y_20_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_dt_params_20 = find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid_dt)
print(f"Mejores hiperparámetros para 20 clases: {best_dt_params_20}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_dt_20 = DecisionTreeClassifier(**best_dt_params_20, random_state=42)
best_dt_20.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_20 = best_dt_20.predict(X_test)
report_20 = classification_report(y_test, y_test_pred_20)
print("\nReporte de Clasificación (Conjunto de Prueba - 20 Clases):")
print(report_20)
dt_results['20_classes'] = {'best_params': best_dt_params_20, 'report': report_20, 'model': best_dt_20}

print("\nResultados de Árbol de Decisión almacenados en el diccionario 'dt_results'.")

In [ ]:
'''
Considerar clasificador:
  - SVM
'''

# Diccionario para almacenar los mejores modelos y reportes de clasificación de SVM
svm_results = {}

# Definir el rango de hiperparámetros a probar (ejemplo basado en hiperparámetros comunes para SVC)
# Si la imagen adjunta especifica otros hiperparámetros o rangos, por favor, házmelo saber.
param_grid_svm = {
    'C': [0.1, 1, 10, 100], # Parámetro de regularización
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'], # Tipo de kernel
    'gamma': ['scale', 'auto', 0.1, 1] # Coeficiente del kernel para 'rbf', 'poly', 'sigmoid'
}

# Función para encontrar los mejores hiperparámetros usando el conjunto de desarrollo
def find_best_svm_params(X_train, y_train, X_dev, y_dev, param_grid):
    best_score = -1
    best_params = None

    # Vamos a probar todas las combinaciones de hiperparámetros
    for C in param_grid['C']:
        for kernel in param_grid['kernel']:
            # Check if gamma is applicable to the current kernel
            if kernel in ['rbf', 'poly', 'sigmoid']:
                for gamma in param_grid['gamma']:
                    # Crear el clasificador con los hiperparámetros actuales
                    svm = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)

                    # Entrenar en el conjunto de entrenamiento
                    svm.fit(X_train, y_train)

                    # Evaluar en el conjunto de desarrollo
                    y_dev_pred = svm.predict(X_dev)
                    accuracy = accuracy_score(y_dev, y_dev_pred)

                    # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                    if accuracy > best_score:
                        best_score = accuracy
                        best_params = {'C': C, 'kernel': kernel, 'gamma': gamma}
            else: # For kernels like 'linear', gamma is not used
                 # Crear el clasificador con los hiperparámetros actuales
                svm = SVC(C=C, kernel=kernel, random_state=42)

                # Entrenar en el conjunto de entrenamiento
                svm.fit(X_train, y_train)

                # Evaluar en el conjunto de desarrollo
                y_dev_pred = svm.predict(X_dev)
                accuracy = accuracy_score(y_dev, y_dev_pred)

                # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                if accuracy > best_score:
                    best_score = accuracy
                    best_params = {'C': C, 'kernel': kernel}


    return best_params

# --- Dataset con 2 clases ---
print("--- Clasificador SVM para 2 Clases ---")
X_train, y_train = X_2_train, y_2_train
X_dev, y_dev = X_2_dev, y_2_dev
X_test, y_test = X_2_test, y_2_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_svm_params_2 = find_best_svm_params(X_train, y_train, X_dev, y_dev, param_grid_svm)
print(f"Mejores hiperparámetros para 2 clases: {best_svm_params_2}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_svm_2 = SVC(**best_svm_params_2, random_state=42)
best_svm_2.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_2 = best_svm_2.predict(X_test)
report_2 = classification_report(y_test, y_test_pred_2)
print("\nReporte de Clasificación (Conjunto de Prueba - 2 Clases):")
print(report_2)
svm_results['2_classes'] = {'best_params': best_svm_params_2, 'report': report_2, 'model': best_svm_2}


# --- Dataset con 5 clases ---
print("\n--- Clasificador SVM para 5 Clases ---")
X_train, y_train = X_5_train, y_5_train
X_dev, y_dev = X_5_dev, y_5_dev
X_test, y_test = X_5_test, y_5_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_svm_params_5 = find_best_svm_params(X_train, y_train, X_dev, y_dev, param_grid_svm)
print(f"Mejores hiperparámetros para 5 clases: {best_svm_params_5}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_svm_5 = SVC(**best_svm_params_5, random_state=42)
best_svm_5.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_5 = best_svm_5.predict(X_test)
report_5 = classification_report(y_test, y_test_pred_5)
print("\nReporte de Clasificación (Conjunto de Prueba - 5 Clases):")
print(report_5)
svm_results['5_classes'] = {'best_params': best_svm_params_5, 'report': report_5, 'model': best_svm_5}


# --- Dataset con 20 clases ---
print("\n--- Clasificador SVM para 20 Clases ---")
X_train, y_train = X_20_train, y_20_train
X_dev, y_dev = X_20_dev, y_20_dev
X_test, y_test = X_20_test, y_20_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_svm_params_20 = find_best_svm_params(X_train, y_train, X_dev, y_dev, param_grid_svm)
print(f"Mejores hiperparámetros para 20 clases: {best_svm_params_20}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_svm_20 = SVC(**best_svm_params_20, random_state=42)
best_svm_20.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_20 = best_svm_20.predict(X_test)
report_20 = classification_report(y_test, y_test_pred_20)
print("\nReporte de Clasificación (Conjunto de Prueba - 20 Clases):")
print(report_20)
svm_results['20_classes'] = {'best_params': best_svm_params_20, 'report': report_20, 'model': best_svm_20}

print("\nResultados de SVM almacenados en el diccionario 'svm_results'.")

In [ ]:
'''
Considerar clasificador:
  - MLP
'''

# Ignorar advertencias de convergencia para simplificar la salida durante el ajuste de hiperparámetros
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Diccionario para almacenar los mejores modelos y reportes de clasificación de MLP
mlp_results = {}

# Definir el rango de hiperparámetros a probar (ejemplo basado en hiperparámetros comunes para MLP)
# Si la imagen adjunta especifica otros hiperparámetros o rangos, por favor, házmelo saber.
param_grid_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)], # Explorar diferentes tamaños de capas simples y dobles
    'activation': ['tanh', 'relu'], # Función de activación para la capa oculta
    'solver': ['adam'], # El optimizador para la optimización de pesos
    'alpha': [0.0001, 0.001], # Parámetro de penalización L2 (término de regularización)
    'learning_rate': ['constant'],
    'learning_rate_init': [0.001, 0.01], # La tasa de aprendizaje inicial utilizada
    'max_iter': [200, 500] # Número máximo de iteraciones
}

# Función para encontrar los mejores hiperparámetros usando el conjunto de desarrollo
def find_best_mlp_params(X_train, y_train, X_dev, y_dev, param_grid):
    best_score = -1
    best_params = None

    # Vamos a probar todas las combinaciones de hiperparámetros
    for hidden_layer_sizes in param_grid['hidden_layer_sizes']:
        for activation in param_grid['activation']:
             for alpha in param_grid['alpha']:
                 for learning_rate in param_grid['learning_rate']:
                     for learning_rate_init in param_grid['learning_rate_init']:
                        # Crear el clasificador con los hiperparámetros actuales
                        mlp = MLPClassifier(
                            hidden_layer_sizes=hidden_layer_sizes,
                            activation=activation,
                            alpha=alpha,
                            learning_rate=learning_rate,
                            learning_rate_init=learning_rate_init, # Incluir learning_rate_init
                            max_iter=1000, # Aumentar max_iter para ayudar a la convergencia
                            random_state=42 # Para reproducibilidad
                        )

                        # Entrenar en el conjunto de entrenamiento
                        mlp.fit(X_train, y_train)

                        # Evaluar en el conjunto de desarrollo
                        y_dev_pred = mlp.predict(X_dev)
                        accuracy = accuracy_score(y_dev, y_dev_pred)

                        # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                        if accuracy > best_score:
                            best_score = accuracy
                            best_params = {
                                'hidden_layer_sizes': hidden_layer_sizes,
                                'activation': activation,
                                'alpha': alpha,
                                'learning_rate': learning_rate,
                                'learning_rate_init': learning_rate_init # Guardar learning_rate_init
                                }

    return best_params

# --- Dataset con 2 clases ---
print("--- Clasificador MLP para 2 Clases ---")
X_train, y_train = X_2_train, y_2_train
X_dev, y_dev = X_2_dev, y_2_dev
X_test, y_test = X_2_test, y_2_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_mlp_params_2 = find_best_mlp_params(X_train, y_train, X_dev, y_dev, param_grid_mlp)
print(f"Mejores hiperparámetros para 2 clases: {best_mlp_params_2}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_mlp_2 = MLPClassifier(**best_mlp_params_2, max_iter=1000, random_state=42)
best_mlp_2.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_2 = best_mlp_2.predict(X_test)
report_2 = classification_report(y_test, y_test_pred_2)
print("\nReporte de Clasificación (Conjunto de Prueba - 2 Clases):")
print(report_2)
mlp_results['2_classes'] = {'best_params': best_mlp_params_2, 'report': report_2, 'model': best_mlp_2}


# --- Dataset con 5 clases ---
print("\n--- Clasificador MLP para 5 Clases ---")
X_train, y_train = X_5_train, y_5_train
X_dev, y_dev = X_5_dev, y_5_dev
X_test, y_test = X_5_test, y_5_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_mlp_params_5 = find_best_mlp_params(X_train, y_train, X_dev, y_dev, param_grid_mlp)
print(f"Mejores hiperparámetros para 5 clases: {best_mlp_params_5}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_mlp_5 = MLPClassifier(**best_mlp_params_5, max_iter=1000, random_state=42)
best_mlp_5.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_5 = best_mlp_5.predict(X_test)
report_5 = classification_report(y_test, y_test_pred_5)
print("\nReporte de Clasificación (Conjunto de Prueba - 5 Clases):")
print(report_5)
mlp_results['5_classes'] = {'best_params': best_mlp_params_5, 'report': report_5, 'model': best_mlp_5}


# --- Dataset con 20 clases ---
print("\n--- Clasificador MLP para 20 Clases ---")
X_train, y_train = X_20_train, y_20_train
X_dev, y_dev = X_20_dev, y_20_dev
X_test, y_test = X_20_test, y_20_test

# Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
best_mlp_params_20 = find_best_mlp_params(X_train, y_train, X_dev, y_dev, param_grid_mlp)
print(f"Mejores hiperparámetros para 20 clases: {best_mlp_params_20}")

# Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
best_mlp_20 = MLPClassifier(**best_mlp_params_20, max_iter=1000, random_state=42)
best_mlp_20.fit(X_train, y_train)

# Evaluar en el conjunto de prueba
y_test_pred_20 = best_mlp_20.predict(X_test)
report_20 = classification_report(y_test, y_test_pred_20)
print("\nReporte de Clasificación (Conjunto de Prueba - 20 Clases):")
print(report_20)
mlp_results['20_classes'] = {'best_params': best_mlp_params_20, 'report': report_20, 'model': best_mlp_20}

print("\nResultados de MLP almacenados en el diccionario 'mlp_results'.")

# Restaurar advertencias
warnings.filterwarnings("default", category=ConvergenceWarning)

In [ ]:
'''
Visualizar las fronteras de decisión de los clasificadores entrenados en los
datasets.
'''

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import os

# Función para graficar las fronteras de decisión
def plot_decision_boundary(model, X, y, title, ax, cmap):
    # Obtener valores mínimos y máximos y agregar un pequeño relleno
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    h = .02  # tamaño del paso en la malla

    # Crear una malla para graficar
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    # Predecir la clase para cada punto en la malla
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Graficar la frontera de decisión
    ax.contourf(xx, yy, Z, cmap=cmap, alpha=.8)

    # Graficar también los puntos de entrenamiento
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, edgecolors='k', s=20)

    ax.set_title(title)
    ax.set_xlabel("Característica 1")
    ax.set_ylabel("Característica 2")
    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    # Agregar una leyenda
    legend = ax.legend(*scatter.legend_elements(), title="Clases")
    ax.add_artist(legend)


# Asumiendo que los diccionarios knn_results, dt_results, svm_results y mlp_results están disponibles
# y contienen los modelos entrenados para '2_classes', '5_classes', y '20_classes'.

datasets = {
    '2_classes': (X_2_test, y_2_test),
    '5_classes': (X_5_test, y_5_test),
    '20_classes': (X_20_test, y_20_test)
}

classifiers = {
    'K-NN': knn_results,
    'Árbol de Decisión': dt_results,
    'SVM': svm_results,
    'MLP': mlp_results
}

# Crear un mapa de colores personalizado para 20 clases si es necesario
# Podrías necesitar un mapa de colores más diverso para 20 clases
cmap_2_classes = ListedColormap(['#FF0000', '#0000FF']) # Rojo, Azul
cmap_5_classes = plt.cm.get_cmap('viridis', 5)
cmap_20_classes = plt.cm.get_cmap('tab20', 20) # tab20 es una buena opción para hasta 20 clases

cmaps = {
    '2_classes': cmap_2_classes,
    '5_classes': cmap_5_classes,
    '20_classes': cmap_20_classes
}


# Graficando individualmente
for dataset_name, (X, y) in datasets.items():
    print(f"\n--- Graficando Fronteras de Decisión para {dataset_name} ---")

    for clf_name, results in classifiers.items():
        if dataset_name in results and 'model' in results[dataset_name]:
            model = results[dataset_name]['model']
            cmap = cmaps[dataset_name]

            fig, ax = plt.subplots(figsize=(6, 6)) # Create a new figure and axes for each plot
            plot_decision_boundary(model, X.values, y.values, f'{clf_name} - {dataset_name}', ax, cmap)

            plt.tight_layout()
            plt.savefig(os.path.join(figs, f'decision_boundary_{clf_name.replace(" ", "_")}_{dataset_name}.png')) # Save the figure with specific naming
            plt.show()
        else:
            print(f"Modelo {clf_name} para {dataset_name} no encontrado. No se generará gráfico.") # Indicate missing plot

print("\nGráficos de fronteras de decisión individuales generados y guardados.")

Escenario 2: grandes datasets vs pequeños datasets

In [ ]:
'''
Generar cinco datasets con las mismas características
(variables deberían ser dos por la naturaleza de los datos),
con cuatro etiquetas/clases y variar sus tamaños de muestra de
acuerdo a 10^2, 10^3, 10^4, 10^5, y 10^6.

NOTA: Los datasets deben ser de tipo 'Gaussian Quartiles'. Son datasets
que se distribuyen como puntos de unplano cartesiano, y cada clase agremia
sus puntos en una región del espacio cartesiano de una manera concéntrica,
cada circulo se aleja de la media, es decir, del origen de coordenadas.
'''

from sklearn.datasets import make_circles
import numpy as np
import pandas as pd

# Función para crear un dataset que se asemeja a 'Gaussian Quartiles' usando make_circles
def create_gaussian_quartiles_dataset(n_samples, random_state=42):
    # Crear círculos anidados usando make_circles
    # Crearemos dos conjuntos de círculos y los combinaremos para representar 4 clases
    # Círculos exteriores
    X_outer, y_outer = make_circles(n_samples=n_samples // 2, factor=0.5, noise=0.0, random_state=random_state) # Ruido eliminado
    # Círculos interiores
    X_inner, y_inner = make_circles(n_samples=n_samples // 2, factor=0.2, noise=0.0, random_state=random_state + 1) # Ruido eliminado y diferente random_state para interiores

    # Combinar los datos
    X = np.vstack((X_outer, X_inner))
    y = np.concatenate((y_outer, y_inner))

    df = pd.DataFrame(X, columns=['feature1', 'feature2'])

    # Asignar clases basadas en los datos combinados para crear 4 'cuartiles'
    # Dividiremos los datos en 4 clases basadas en el ángulo/cuadrante para una sensación más de 'cuartil',
    # mientras mantenemos la naturaleza concéntrica de make_circles.
    # Esta es una forma de interpretar "Gaussian Quartiles" con 4 clases a partir de datos concéntricos.
    # Un enfoque más simple es usar las etiquetas originales 0/1 de make_circles para dos capas,
    # y luego potencialmente dividir cada capa basada en el cuadrante si se necesita una estructura de 'cuartil' de 4 clases verdadera.

    # Intentemos asignar clases basadas en las etiquetas originales y una división simple (por ejemplo, por el eje x)
    # Esto puede no representar perfectamente los 'Gaussian Quartiles', pero cumple con el requisito de 4 clases
    # y utiliza la naturaleza concéntrica de make_circles.

    df['class'] = -1 # Inicializar con un marcador de posición

    # Clase 0: Círculo exterior, feature1 < 0
    df.loc[(y == 0) & (df['feature1'] < 0), 'class'] = 0
    # Clase 1: Círculo exterior, feature1 >= 0
    df.loc[(y == 0) & (df['feature1'] >= 0), 'class'] = 1
    # Clase 2: Círculo interior, feature1 < 0
    df.loc[(y == 1) & (df['feature1'] < 0), 'class'] = 2
    # Clase 3: Círculo interior, feature1 >= 0
    df.loc[(y == 1) & (df['feature1'] >= 0), 'class'] = 3

    # Manejar cualquier muestra que no encaje perfectamente (por ejemplo, exactamente en el eje x) - asignar a una clase vecina
    df['class'] = df['class'].replace(-1, 3) # Alternativa simple: asignar a la clase 3


    return df

# Generar datasets con diferentes tamaños de muestra
sample_sizes = [10**2, 10**3, 10**4, 10**5, 10**6]
datasets_size = {}

for size in sample_sizes:
    print(f"Generando dataset con {size} muestras...")
    df = create_gaussian_quartiles_dataset(n_samples=size, random_state=42)
    datasets_size[f'{size}_samples'] = df
    print(f"Dataset con {size} muestras generado. Forma: {df.shape}")
    print(f"Distribución de clases para {size} muestras:")
    display(df['class'].value_counts().sort_index())
    print("-" * 30)

print("\nDatasets con diferentes tamaños de muestra generados y almacenados en 'datasets_size'.")

In [ ]:
'''
Dividir cada dataset generado en conjuntos de entrenamiento, desarrollo y prueba.
Conservar la proporción de clases.
'''

# Diccionario para almacenar los conjuntos de datos divididos para cada tamaño de muestra
split_datasets = {}

for size_label, df in datasets_size.items():
    print(f"Dividiendo dataset para {size_label}...")

    X = df[['feature1', 'feature2']]
    y = df['class']

    # Dividir en entrenamiento (60%) y un conjunto temporal (40% para desarrollo + prueba)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.4, random_state=42, stratify=y
    )

    # Dividir el conjunto temporal (40%) en desarrollo (20%) y prueba (20%)
    X_dev, X_test, y_dev, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    # Almacenar los conjuntos divididos en el diccionario
    split_datasets[size_label] = {
        'X_train': X_train, 'y_train': y_train,
        'X_dev': X_dev, 'y_dev': y_dev,
        'X_test': X_test, 'y_test': y_test
    }

    print(f"Dataset para {size_label} dividido.")
    print(f"  Conjunto de entrenamiento: {X_train.shape}")
    print(f"  Conjunto de desarrollo: {X_dev.shape}")
    print(f"  Conjunto de prueba: {X_test.shape}")
    print("-" * 30)

print("\nDatasets divididos y almacenados en el diccionario 'split_datasets'.")

In [ ]:
'''
Considerar clasificador K-NN para los datasets construidos. Jugar con el numero
de vecinos (hiperparámetro),y elegir el mejor, luego, mostrar el reporte de
clasificación del mejor modelo de cada dataset.
'''

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# Diccionario para almacenar los mejores modelos y reportes de clasificación por tamaño de dataset
knn_size_results = {}

# Definir el rango de vecinos a probar
k_range = range(1, 31)

# Iterar sobre cada dataset dividido por tamaño
for size_label, data in split_datasets.items():
    print(f"\n--- Clasificador K-NN para {size_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    best_k = None
    best_accuracy = -1

    # Paso 1 y 2: Entrenar en el conjunto de entrenamiento y encontrar el mejor n_neighbors usando el conjunto de desarrollo
    print("Buscando el mejor n_neighbors usando el conjunto de desarrollo...")
    for k in k_range:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train) # Entrenar en el conjunto de entrenamiento
        y_dev_pred = knn.predict(X_dev) # Predecir en el conjunto de desarrollo
        accuracy = accuracy_score(y_dev, y_dev_pred) # Evaluar en el conjunto de desarrollo

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_k = k

    print(f"Mejor n_neighbors para {size_label}: {best_k}")

    # Paso 3: Entrenar el modelo final en el conjunto de entrenamiento con el best k encontrado en el conjunto de desarrollo y evaluar en el conjunto de prueba
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con el mejor k={best_k} y evaluando en el conjunto de prueba...")
    best_knn = KNeighborsClassifier(n_neighbors=best_k)
    best_knn.fit(X_train, y_train)

    # Evaluar en el conjunto de prueba
    y_test_pred = best_knn.predict(X_test)
    report = classification_report(y_test, y_test_pred)
    print(f"\nReporte de Clasificación (Conjunto de Prueba - {size_label}):")
    print(report)
    knn_size_results[size_label] = {'best_k': best_k, 'report': report, 'model': best_knn}

print("\nResultados de K-NN por tamaño de dataset almacenados en el diccionario 'knn_size_results'.")

In [ ]:
'''
Considerar clasificador arbol de decisión para los datasets construidos. Probar
con diferentes hiperparámetros, y elegir la mejor configuración de ellos, luego,
mostrar el reporte de clasificación del mejor modelo de cada dataset.
'''

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Diccionario para almacenar los mejores modelos y reportes de clasificación del Árbol de Decisión por tamaño de dataset
dt_size_results = {}

# Definir el rango de hiperparámetros a probar según la imagen adjunta:
# criterion: gini, entropy
# max_depth: 1, 2, 3, 4, 5
# min_samples_split: 2, 3, 4, 5
# min_samples_leaf: 1, 2, 3, 4, 5
param_grid_dt = {
    'criterion': ['gini'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Función para encontrar los mejores hiperparámetros usando el conjunto de desarrollo
def find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid):
    best_score = -1
    best_params = None

    # Vamos a probar todas las combinaciones de hiperparámetros
    for criterion in param_grid['criterion']:
        for max_depth in param_grid['max_depth']:
            for min_samples_split in param_grid['min_samples_split']:
                for min_samples_leaf in param_grid['min_samples_leaf']:
                    # Crear el clasificador con los hiperparámetros actuales
                    dt = DecisionTreeClassifier(
                        criterion=criterion,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        random_state=42 # Para reproducibilidad
                    )

                    # Entrenar en el conjunto de entrenamiento
                    dt.fit(X_train, y_train)

                    # Evaluar en el conjunto de desarrollo
                    y_dev_pred = dt.predict(X_dev)
                    accuracy = accuracy_score(y_dev, y_dev_pred)

                    # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                    if accuracy > best_score:
                        best_score = accuracy
                        best_params = {
                            'criterion': criterion,
                            'max_depth': max_depth,
                            'min_samples_split': min_samples_split,
                            'min_samples_leaf': min_samples_leaf
                            }

    return best_params

# Iterar sobre cada dataset dividido por tamaño
for size_label, data in split_datasets.items():
    print(f"\n--- Clasificador Árbol de Decisión para {size_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    # Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
    print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
    best_dt_params = find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid_dt)
    print(f"Mejores hiperparámetros para {size_label}: {best_dt_params}")

    # Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
    best_dt = DecisionTreeClassifier(**best_dt_params, random_state=42)
    best_dt.fit(X_train, y_train)

    # Evaluar en el conjunto de prueba
    y_test_pred = best_dt.predict(X_test)
    report = classification_report(y_test, y_test_pred)
    print(f"\nReporte de Clasificación (Conjunto de Prueba - {size_label}):")
    print(report)
    dt_size_results[size_label] = {'best_params': best_dt_params, 'report': report, 'model': best_dt}

print("\nResultados de Árbol de Decisión por tamaño de dataset almacenados en el diccionario 'dt_size_results'.")

In [ ]:
'''
Considerar clasificador SVM para los datasets construidos. Probar con diferentes
hiperparámetros,y elegir la mejor configuración de ellos, luego, mostrar el
reporte de clasificación del mejor modelo de cada dataset.
'''

# Diccionario para almacenar los mejores modelos y reportes de clasificación de SVM por tamaño de dataset
svm_size_results = {}

# Definir el rango de hiperparámetros a probar según la imagen adjunta:
# C: 0.1, 1, 10, 100
# kernel: linear, poly, rbf, sigmoid
# gamma: scale, auto, 0.1, 1
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# Función para encontrar los mejores hiperparámetros usando el conjunto de desarrollo con límite de tiempo
def find_best_svm_params_timed(X_train, y_train, X_dev, y_dev, param_grid, time_limit_per_fit=300): # time_limit_per_fit in seconds (5 minutes)
    best_score = -1
    best_params = None
    total_start_time = time.time()

    # Vamos a probar todas las combinaciones de hiperparámetros
    for C in param_grid['C']:
        for kernel in param_grid['kernel']:
            # Check if gamma is applicable to the current kernel
            if kernel in ['rbf', 'poly', 'sigmoid']:
                for gamma in param_grid['gamma']:
                    # Check if overall time limit is exceeded
                    if time.time() - total_start_time > time_limit_per_fit:
                        print("Tiempo total excedido durante la búsqueda de hiperparámetros. Terminando búsqueda.")
                        return best_params # Return the best found so far

                    # Crear el clasificador con los hiperparámetros actuales
                    svm = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)

                    # Entrenar en el conjunto de entrenamiento con límite de tiempo
                    fit_start_time = time.time()
                    try:
                        svm.fit(X_train, y_train)
                        fit_time = time.time() - fit_start_time
                        if fit_time > time_limit_per_fit:
                            print(f"Entrenamiento de SVM con params C={C}, kernel={kernel}, gamma={gamma} excedió el límite de tiempo ({fit_time:.2f}s). Saltando a la siguiente combinación.")
                            continue # Skip evaluation if fit took too long

                        # Evaluar en el conjunto de desarrollo
                        y_dev_pred = svm.predict(X_dev)
                        accuracy = accuracy_score(y_dev, y_dev_pred)

                        # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                        if accuracy > best_score:
                            best_score = accuracy
                            best_params = {'C': C, 'kernel': kernel, 'gamma': gamma}
                            print(f"Nuevo mejor score en dev: {best_score:.4f} con params: {best_params}")

                    except Exception as e:
                        print(f"Error durante el entrenamiento o evaluación de SVM con params C={C}, kernel={kernel}, gamma={gamma}: {e}")
                        continue # Continue to the next combination if there's an error

            else: # For kernels like 'linear', gamma is not used
                 # Check if overall time limit is exceeded
                if time.time() - total_start_time > time_limit_per_fit:
                    print("Tiempo total excedido durante la búsqueda de hiperparámetros. Terminando búsqueda.")
                    return best_params # Return the best found so far

                 # Crear el clasificador con los hiperparámetros actuales
                svm = SVC(C=C, kernel=kernel, random_state=42)

                # Entrenar en el conjunto de entrenamiento con límite de tiempo
                fit_start_time = time.time()
                try:
                    svm.fit(X_train, y_train)
                    fit_time = time.time() - fit_start_time
                    if fit_time > time_limit_per_fit:
                        print(f"Entrenamiento de SVM con params C={C}, kernel={kernel} excedió el límite de tiempo ({fit_time:.2f}s). Saltando a la siguiente combinación.")
                        continue # Skip evaluation if fit took too long

                    # Evaluar en el conjunto de desarrollo
                    y_dev_pred = svm.predict(X_dev)
                    accuracy = accuracy_score(y_dev, y_dev_pred)

                    # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                    if accuracy > best_score:
                        best_score = accuracy
                        best_params = {'C': C, 'kernel': kernel}
                        print(f"Nuevo mejor score en dev: {best_score:.4f} con params: {best_params}")

                except Exception as e:
                    print(f"Error durante el entrenamiento o evaluación de SVM con params C={C}, kernel={kernel}: {e}")
                    continue # Continue to the next combination if there's an error


    return best_params

# Iterar sobre cada dataset dividido por tamaño
for size_label, data in split_datasets.items():
    print(f"\n--- Clasificador SVM para {size_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    # Encontrar los mejores hiperparámetros usando el conjunto de desarrollo con límite de tiempo
    print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo (con límite de tiempo de 5 minutos por ajuste)...")
    best_svm_params = find_best_svm_params_timed(X_train, y_train, X_dev, y_dev, param_grid_svm, time_limit_per_fit=300)

    if best_svm_params is None:
        print(f"No se encontraron mejores hiperparámetros para {size_label} dentro del límite de tiempo.")
        svm_size_results[size_label] = {'best_params': None, 'report': "Búsqueda de hiperparámetros no completada.", 'model': None}
        continue # Skip to the next dataset size

    print(f"Mejores hiperparámetros para {size_label}: {best_svm_params}")

    # Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
    best_svm = SVC(**best_svm_params, random_state=42)

    fit_start_time = time.time()
    try:
        best_svm.fit(X_train, y_train)
        fit_time = time.time() - fit_start_time
        if fit_time > 300: # Check again for the final model training
             print(f"Entrenamiento del modelo SVM final para {size_label} excedió el límite de tiempo ({fit_time:.2f}s). No se generará reporte.")
             svm_size_results[size_label] = {'best_params': best_svm_params, 'report': "Entrenamiento final excedió el límite de tiempo.", 'model': None}
             continue # Skip evaluation if final fit took too long


        # Evaluar en el conjunto de prueba
        y_test_pred = best_svm.predict(X_test)
        report = classification_report(y_test, y_test_pred)
        print(f"\nReporte de Clasificación (Conjunto de Prueba - {size_label}):")
        print(report)
        svm_size_results[size_label] = {'best_params': best_svm_params, 'report': report, 'model': best_svm}

    except Exception as e:
        print(f"Error durante el entrenamiento o evaluación del modelo SVM final para {size_label}: {e}")
        svm_size_results[size_label] = {'best_params': best_svm_params, 'report': f"Error: {e}", 'model': None}


print("\nResultados de SVM por tamaño de dataset almacenados en el diccionario 'svm_size_results'.")

In [ ]:
'''
Considerar clasificador MLP para los datasets construidos. Probar con diferentes
hiperparámetros,y elegir la mejor configuración de ellos, luego, mostrar el reporte
de clasificación del mejor modelo de cada dataset.
'''

# Ignorar advertencias de convergencia para simplificar la salida durante el ajuste de hiperparámetros
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Diccionario para almacenar los mejores modelos y reportes de clasificación de MLP por tamaño de dataset
mlp_size_results = {}

# Definir el rango de hiperparámetros
param_grid_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'activation': ['tanh', 'relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01]
}

# Function to find the best hyperparameters using the development set with a time limit
def find_best_mlp_params_timed(X_train, y_train, X_dev, y_dev, param_grid, time_limit_per_fit=300): # time_limit_per_fit in seconds (5 minutes)
    best_score = -1
    best_params = None
    total_start_time = time.time()

    # Vamos a probar todas las combinaciones de hiperparámetros
    for hidden_layer_sizes in param_grid['hidden_layer_sizes']:
        for activation in param_grid['activation']:
            for alpha in param_grid['alpha']:
                for learning_rate_init in param_grid['learning_rate_init']:
                    # Check if overall time limit is exceeded
                    if time.time() - total_start_time > time_limit_per_fit:
                        print("Tiempo total excedido durante la búsqueda de hiperparámetros. Terminando búsqueda.")
                        return best_params # Return the best found so far

                    # Crear el clasificador con los hiperparámetros actuales
                    mlp = MLPClassifier(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation=activation,
                        solver='adam', # Fixed solver as per image
                        alpha=alpha,
                        learning_rate='constant', # Fixed learning_rate as per image
                        learning_rate_init=learning_rate_init,
                        max_iter=500, # Use a reasonable max_iter for tuning
                        random_state=42 # For reproducibility
                    )

                    # Entrenar en el conjunto de entrenamiento with time limit
                    fit_start_time = time.time()
                    try:
                        mlp.fit(X_train, y_train)
                        fit_time = time.time() - fit_start_time
                        if fit_time > time_limit_per_fit:
                            print(f"Entrenamiento de MLP con params {mlp.get_params()} excedió el límite de tiempo ({fit_time:.2f}s). Saltando a la siguiente combinación.")
                            continue # Skip evaluation if fit took too long

                        # Evaluar en el conjunto de desarrollo
                        y_dev_pred = mlp.predict(X_dev)
                        accuracy = accuracy_score(y_dev, y_dev_pred)

                        # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                        if accuracy > best_score:
                            best_score = accuracy
                            best_params = {
                                'hidden_layer_sizes': hidden_layer_sizes,
                                'activation': activation,
                                'solver': 'adam',
                                'alpha': alpha,
                                'learning_rate': 'constant',
                                'learning_rate_init': learning_rate_init
                                }
                            print(f"Nuevo mejor score en dev: {best_score:.4f} con params: {best_params}")

                    except Exception as e:
                        print(f"Error durante el entrenamiento o evaluación de MLP con params {mlp.get_params()}: {e}")
                        continue # Continue to the next combination if there's an error


    return best_params


# Iterar sobre cada dataset dividido por tamaño
for size_label, data in split_datasets.items():
    print(f"\n--- Clasificador MLP para {size_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    # Encontrar los mejores hiperparámetros usando el conjunto de desarrollo con límite de tiempo
    print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo (con límite de tiempo de 5 minutos por ajuste)...")
    best_mlp_params = find_best_mlp_params_timed(X_train, y_train, X_dev, y_dev, param_grid_mlp, time_limit_per_fit=300)

    if best_mlp_params is None:
        print(f"No se encontraron mejores hiperparámetros para {size_label} dentro del límite de tiempo.")
        mlp_size_results[size_label] = {'best_params': None, 'report': "Búsqueda de hiperparámetros no completada.", 'model': None}
        continue # Skip to the next dataset size


    print(f"Mejores hiperparámetros para {size_label}: {best_mlp_params}")

    # Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
    # Use a higher max_iter for the final training for better convergence
    best_mlp = MLPClassifier(**best_mlp_params, max_iter=1000, random_state=42)

    fit_start_time = time.time()
    try:
        best_mlp.fit(X_train, y_train)
        fit_time = time.time() - fit_start_time
        if fit_time > 300: # Check again for the final model training
             print(f"Entrenamiento del modelo MLP final para {size_label} excedió el límite de tiempo ({fit_time:.2f}s). No se generará reporte.")
             mlp_size_results[size_label] = {'best_params': best_mlp_params, 'report': "Entrenamiento final excedió el límite de tiempo.", 'model': None}
             continue # Skip evaluation if final fit took too long


        # Evaluar en el conjunto de prueba
        y_test_pred = best_mlp.predict(X_test)
        report = classification_report(y_test, y_test_pred)
        print(f"\nReporte de Clasificación (Conjunto de Prueba - {size_label}):")
        print(report)
        mlp_size_results[size_label] = {'best_params': best_mlp_params, 'report': report, 'model': best_mlp}

    except Exception as e:
        print(f"Error durante el entrenamiento o evaluación del modelo MLP final para {size_label}: {e}")
        mlp_size_results[size_label] = {'best_params': best_mlp_params, 'report': f"Error: {e}", 'model': None}


print("\nResultados de MLP por tamaño de dataset almacenados en el diccionario 'mlp_size_results'.")

# Restaurar advertencias
warnings.filterwarnings("default", category=ConvergenceWarning)


--- Clasificador MLP para 100_samples ---
Buscando los mejores hiperparámetros usando el conjunto de desarrollo (con límite de tiempo de 5 minutos por ajuste)...
Nuevo mejor score en dev: 0.9500 con params: {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0001, 'learning_rate': 'constant', 'learning_rate_init': 0.001}
Nuevo mejor score en dev: 1.0000 con params: {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0001, 'learning_rate': 'constant', 'learning_rate_init': 0.01}
Mejores hiperparámetros para 100_samples: {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0001, 'learning_rate': 'constant', 'learning_rate_init': 0.01}
Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...

Reporte de Clasificación (Conjunto de Prueba - 100_samples):
              precision    recall  f1-score   support

           0       1.00      

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
'''
Visualización de resultados por medio de un barplot
'''

# Function to extract metrics from classification reports
def extract_metrics(results_dict):
    metrics_data = []
    for size_label, result in results_dict.items():
        if 'report' in result and result['report'] != "Búsqueda de hiperparámetros no completada." and result['report'] != "Entrenamiento final excedió el límite de tiempo." and result['report'] is not None:
            report_lines = result['report'].split('\n')
            # Find the line with macro avg or weighted avg
            macro_avg_line = None
            weighted_avg_line = None
            for line in report_lines:
                if 'macro avg' in line:
                    macro_avg_line = line
                if 'weighted avg' in line:
                    weighted_avg_line = line

            if weighted_avg_line: # Prefer weighted avg if available
                parts = weighted_avg_line.split()
                # Extract metrics (precision, recall, f1-score, support)
                # Ensure there are enough parts before accessing
                if len(parts) >= 5:
                    metrics = {
                        'precision': float(parts[2]),
                        'recall': float(parts[3]),
                        'f1-score': float(parts[4]),
                        'support': int(parts[5])
                    }
                    metrics_data.append({'dataset_size': size_label, **metrics})
            elif macro_avg_line: # Fallback to macro avg
                 parts = macro_avg_line.split()
                 if len(parts) >= 5:
                    metrics = {
                        'precision': float(parts[2]),
                        'recall': float(parts[3]),
                        'f1-score': float(parts[4]),
                        'support': int(parts[5])
                    }
                    metrics_data.append({'dataset_size': size_label, **metrics})


    return pd.DataFrame(metrics_data)

# Extract metrics for each classifier (using the size results dictionaries)
knn_metrics_size = extract_metrics(knn_size_results)
dt_metrics_size = extract_metrics(dt_size_results)
svm_metrics_size = extract_metrics(svm_size_results)
mlp_metrics_size = extract_metrics(mlp_size_results)

# Combine metrics into a single DataFrame for easier plotting
all_metrics_size = pd.concat([
    knn_metrics_size.assign(classifier='K-NN'),
    dt_metrics_size.assign(classifier='Árbol de Decisión'),
    svm_metrics_size.assign(classifier='SVM'),
    mlp_metrics_size.assign(classifier='MLP')
])

# Define the order of dataset sizes for plotting
size_order = ['100_samples', '1000_samples', '10000_samples', '100000_samples', '1000000_samples']
all_metrics_size['dataset_size'] = pd.Categorical(all_metrics_size['dataset_size'], categories=size_order, ordered=True)
all_metrics_size = all_metrics_size.sort_values('dataset_size')


# Plotting the results
metrics_to_plot = ['precision', 'recall', 'f1-score']

for metric in metrics_to_plot:
    plt.figure(figsize=(12, 7)) # Adjust figure size for more data points
    barplot = sns.barplot(x='dataset_size', y=metric, hue='classifier', data=all_metrics_size, palette='viridis')
    plt.title(f'{metric.capitalize()} across Dataset Sizes for Different Classifiers')
    plt.xlabel('Dataset Size')
    plt.ylabel(metric.capitalize())
    plt.ylim(0, 1.1) # Set y-axis limit for better comparison
    plt.legend(title='Classifier')
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability

    # Add text labels for the bars (optional, can make plot crowded with many bars)
    # for container in barplot.containers:
    #     barplot.bar_label(container, fmt='%.2f', label_type='edge')

    plt.tight_layout()
    # Ensure 'figs' is accessible or defined - assuming it's defined in an earlier cell and accessible
    plt.savefig(os.path.join(figs, f'size_comparison_{metric}_barplot.png')) # Save the figure
    plt.show()

print("\nBar plots generated and saved for Precision, Recall, and F1-score across different dataset sizes.")

Escenario 3: dataset limpio vs dataset ruidoso

In [ ]:
'''
Generación de un dataset binario limpio y dos versiones ruidosas con
diferentes tasas de corrupción (10% y 30%).

Los datos deben ser de tipo 'Mooon'
'''

# Generate a clean binary dataset using make_moons
n_samples = 700
X_clean, y_clean = make_moons(n_samples=n_samples, noise=0, random_state=42)

df_clean = pd.DataFrame(X_clean, columns=['feature1', 'feature2'])
df_clean['class'] = y_clean

print("Clean Dataset (2 Classes - Moons):")
display(df_clean.head())
print("\nClass distribution in the clean dataset:")
display(df_clean['class'].value_counts())

# Function to introduce noise into the dataset (flipping labels)
def introduce_noise(y, noise_rate, random_state=42):
    y_noisy = y.copy()
    n_samples = len(y)
    n_noisy_samples = int(n_samples * noise_rate)

    # Get indices to flip
    noisy_indices = np.random.choice(n_samples, size=n_noisy_samples, replace=False)

    # Flip the labels for the selected indices
    for i in noisy_indices:
        # Assuming binary classification (0 and 1), flip the label
        y_noisy[i] = 1 - y_noisy[i]
    return y_noisy

# Generate dataset with 10% noise
y_noisy_10 = introduce_noise(y_clean, noise_rate=0.1, random_state=42)
df_noisy_10 = df_clean.copy()
df_noisy_10['class'] = y_noisy_10

print("\nDataset with 10% Noise (2 Classes - Moons):")
display(df_noisy_10.head())
print("\nClass distribution in the dataset with 10% noise:")
display(df_noisy_10['class'].value_counts())


# Generate dataset with 30% noise
y_noisy_30 = introduce_noise(y_clean, noise_rate=0.3, random_state=42)
df_noisy_30 = df_clean.copy()
df_noisy_30['class'] = y_noisy_30

print("\nDataset with 30% Noise (2 Classes - Moons):")
display(df_noisy_30.head())
print("\nClass distribution in the dataset with 30% noise:")
display(df_noisy_30['class'].value_counts())

# Store the datasets in a dictionary for easier access later
datasets_noise = {
    'clean': df_clean,
    'noisy_10': df_noisy_10,
    'noisy_30': df_noisy_30
}

In [ ]:
'''
Dividir cada dataset generado en conjuntos de entrenamiento, desarrollo y prueba.
Conservar la proporción de clases.
'''

# Diccionario para almacenar los conjuntos de datos divididos para cada dataset (limpio/ruidoso)
split_datasets_noise = {}

for noise_label, df in datasets_noise.items():
    print(f"Dividiendo dataset: {noise_label}...")

    X = df[['feature1', 'feature2']]
    y = df['class']

    # Dividir en entrenamiento (60%) y un conjunto temporal (40% para desarrollo + prueba)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.4, random_state=42, stratify=y
    )

    # Dividir el conjunto temporal (40%) en desarrollo (20%) y prueba (20%)
    X_dev, X_test, y_dev, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    # Almacenar los conjuntos divididos en el diccionario
    split_datasets_noise[noise_label] = {
        'X_train': X_train, 'y_train': y_train,
        'X_dev': X_dev, 'y_dev': y_dev,
        'X_test': X_test, 'y_test': y_test
    }

    print(f"Dataset {noise_label} dividido.")
    print(f"  Conjunto de entrenamiento: {X_train.shape}")
    print(f"  Conjunto de desarrollo: {X_dev.shape}")
    print(f"  Conjunto de prueba: {X_test.shape}")
    print("-" * 30)

print("\nDatasets (limpio/ruidoso) divididos y almacenados en el diccionario 'split_datasets_noise'.")

In [ ]:
'''
Considerar clasificador K-NN para los datasets construidos. Jugar con el numero
de vecinos (hiperparámetro),y elegir el mejor, luego, mostrar el reporte de
clasificación del mejor modelo de cada dataset.
'''

# Diccionario para almacenar los mejores modelos y reportes de clasificación por nivel de ruido
knn_noise_results = {}

# Definir el rango de vecinos a probar (can be adjusted if needed)
k_range = range(1, 31) # Using the same range as in the previous K-NN task

# Iterar sobre cada dataset dividido (limpio/ruidoso)
for noise_label, data in split_datasets_noise.items():
    print(f"\n--- Clasificador K-NN para el dataset {noise_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    best_k = None
    best_accuracy = -1

    # Paso 1 y 2: Entrenar en el conjunto de entrenamiento y encontrar el mejor n_neighbors usando el conjunto de desarrollo
    print("Buscando el mejor n_neighbors usando el conjunto de desarrollo...")
    for k in k_range:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train) # Entrenar en el conjunto de entrenamiento
        y_dev_pred = knn.predict(X_dev) # Predecir en el conjunto de desarrollo
        accuracy = accuracy_score(y_dev, y_dev_pred) # Evaluar en el conjunto de desarrollo

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_k = k

    print(f"Mejor n_neighbors para el dataset {noise_label}: {best_k}")

    # Paso 3: Entrenar el modelo final en el conjunto de entrenamiento con el best k encontrado en el conjunto de desarrollo y evaluar en el conjunto de prueba
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con el mejor k={best_k} y evaluando en el conjunto de prueba...")
    best_knn = KNeighborsClassifier(n_neighbors=best_k)
    best_knn.fit(X_train, y_train)

    # Evaluar en el conjunto de prueba
    y_test_pred = best_knn.predict(X_test)
    report = classification_report(y_test, y_test_pred)
    print(f"\nReporte de Clasificación (Conjunto de Prueba - Dataset {noise_label}):")
    print(report)
    knn_noise_results[noise_label] = {'best_k': best_k, 'report': report, 'model': best_knn}

print("\nResultados de K-NN por nivel de ruido almacenados en el diccionario 'knn_noise_results'.")

In [ ]:
'''
Considerar clasificador arbol de decisión para los datasets construidos. Probar
con diferentes hiperparámetros, y elegir la mejor configuración de ellos, luego,
mostrar el reporte de clasificación del mejor modelo de cada dataset.
'''

# Diccionario para almacenar los mejores modelos y reportes de clasificación del Árbol de Decisión por nivel de ruido
dt_noise_results = {}

# Definir el rango de hiperparámetros a probar (using the same grid as in the previous DT task)
param_grid_dt = {
    'criterion': ['gini'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Función para encontrar los mejores hiperparámetros usando el conjunto de desarrollo
def find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid):
    best_score = -1
    best_params = None

    # Vamos a probar todas las combinaciones de hiperparámetros
    for criterion in param_grid['criterion']:
        for max_depth in param_grid['max_depth']:
            for min_samples_split in param_grid['min_samples_split']:
                for min_samples_leaf in param_grid['min_samples_leaf']:
                    # Crear el clasificador con los hiperparámetros actuales
                    dt = DecisionTreeClassifier(
                        criterion=criterion,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        random_state=42 # Para reproducibilidad
                    )

                    # Entrenar en el conjunto de entrenamiento
                    dt.fit(X_train, y_train)

                    # Evaluar en el conjunto de desarrollo
                    y_dev_pred = dt.predict(X_dev)
                    accuracy = accuracy_score(y_dev, y_dev_pred)

                    # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                    if accuracy > best_score:
                        best_score = accuracy
                        best_params = {
                            'criterion': criterion,
                            'max_depth': max_depth,
                            'min_samples_split': min_samples_split,
                            'min_samples_leaf': min_samples_leaf
                            }

    return best_params

# Iterar sobre cada dataset dividido (limpio/ruidoso)
for noise_label, data in split_datasets_noise.items():
    print(f"\n--- Clasificador Árbol de Decisión para el dataset {noise_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    # Encontrar los mejores hiperparámetros usando el conjunto de desarrollo
    print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo...")
    best_dt_params = find_best_dt_params(X_train, y_train, X_dev, y_dev, param_grid_dt)
    print(f"Mejores hiperparámetros para el dataset {noise_label}: {best_dt_params}")

    # Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
    best_dt = DecisionTreeClassifier(**best_dt_params, random_state=42)
    best_dt.fit(X_train, y_train)

    # Evaluar en el conjunto de prueba
    y_test_pred = best_dt.predict(X_test)
    report = classification_report(y_test, y_test_pred)
    print(f"\nReporte de Clasificación (Conjunto de Prueba - Dataset {noise_label}):")
    print(report)
    dt_noise_results[noise_label] = {'best_params': best_dt_params, 'report': report, 'model': best_dt}

print("\nResultados de Árbol de Decisión por nivel de ruido almacenados en el diccionario 'dt_noise_results'.")

In [ ]:
'''
Considerar clasificador SVM para los datasets construidos. Probar con diferentes
hiperparámetros,y elegir la mejor configuración de ellos, luego, mostrar el
reporte de clasificación del mejor modelo de cada dataset.
'''

# Diccionario para almacenar los mejores modelos y reportes de clasificación de SVM por nivel de ruido
svm_noise_results = {}

# Definir el rango de hiperparámetros a probar (using the same grid as in the previous SVM task)
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# Función para encontrar los mejores hiperparámetros usando el conjunto de desarrollo con límite de tiempo
def find_best_svm_params_timed(X_train, y_train, X_dev, y_dev, param_grid, time_limit_per_fit=300): # time_limit_per_fit in seconds (5 minutes)
    best_score = -1
    best_params = None
    total_start_time = time.time()

    # Vamos a probar todas las combinaciones de hiperparámetros
    for C in param_grid['C']:
        for kernel in param_grid['kernel']:
            # Check if gamma is applicable to the current kernel
            if kernel in ['rbf', 'poly', 'sigmoid']:
                for gamma in param_grid['gamma']:
                    # Check if overall time limit is exceeded
                    if time.time() - total_start_time > time_limit_per_fit:
                        print("Tiempo total excedido durante la búsqueda de hiperparámetros. Terminando búsqueda.")
                        return best_params # Return the best found so far

                    # Crear el clasificador con los hiperparámetros actuales
                    svm = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)

                    # Entrenar en el conjunto de entrenamiento con límite de tiempo
                    fit_start_time = time.time()
                    try:
                        svm.fit(X_train, y_train)
                        fit_time = time.time() - fit_start_time
                        if fit_time > time_limit_per_fit:
                            print(f"Entrenamiento de SVM con params C={C}, kernel={kernel}, gamma={gamma} excedió el límite de tiempo ({fit_time:.2f}s). Saltando a la siguiente combinación.")
                            continue # Skip evaluation if fit took too long

                        # Evaluar en el conjunto de desarrollo
                        y_dev_pred = svm.predict(X_dev)
                        accuracy = accuracy_score(y_dev, y_dev_pred)

                        # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                        if accuracy > best_score:
                            best_score = accuracy
                            best_params = {'C': C, 'kernel': kernel, 'gamma': gamma}
                            print(f"Nuevo mejor score en dev: {best_score:.4f} con params: {best_params}")

                    except Exception as e:
                        print(f"Error durante el entrenamiento o evaluación de SVM con params C={C}, kernel={kernel}, gamma={gamma}: {e}")
                        continue # Continue to the next combination if there's an error

            else: # For kernels like 'linear', gamma is not used
                 # Check if overall time limit is exceeded
                if time.time() - total_start_time > time_limit_per_fit:
                    print("Tiempo total excedido durante la búsqueda de hiperparámetros. Terminando búsqueda.")
                    return best_params # Return the best found so far

                 # Crear el clasificador con los hiperparámetros actuales
                svm = SVC(C=C, kernel=kernel, random_state=42)

                # Entrenar en el conjunto de entrenamiento con límite de tiempo
                fit_start_time = time.time()
                try:
                    svm.fit(X_train, y_train)
                    fit_time = time.time() - fit_start_time
                    if fit_time > time_limit_per_fit:
                        print(f"Entrenamiento de SVM con params C={C}, kernel={kernel} excedió el límite de tiempo ({fit_time:.2f}s). Saltando a la siguiente combinación.")
                        continue # Skip evaluation if fit took too long

                    # Evaluar en el conjunto de desarrollo
                    y_dev_pred = svm.predict(X_dev)
                    accuracy = accuracy_score(y_dev, y_dev_pred)

                    # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                    if accuracy > best_score:
                        best_score = accuracy
                        best_params = {'C': C, 'kernel': kernel}
                        print(f"Nuevo mejor score en dev: {best_score:.4f} con params: {best_params}")

                except Exception as e:
                    print(f"Error durante el entrenamiento o evaluación de SVM con params C={C}, kernel={kernel}: {e}")
                    continue # Continue to the next combination if there's an error


    return best_params

# Iterar sobre cada dataset dividido por nivel de ruido
for noise_label, data in split_datasets_noise.items():
    print(f"\n--- Clasificador SVM para el dataset {noise_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    # Encontrar los mejores hiperparámetros usando el conjunto de desarrollo con límite de tiempo
    print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo (con límite de tiempo de 5 minutos por ajuste)...")
    best_svm_params = find_best_svm_params_timed(X_train, y_train, X_dev, y_dev, param_grid_svm, time_limit_per_fit=300)

    if best_svm_params is None:
        print(f"No se encontraron mejores hiperparámetros para el dataset {noise_label} dentro del límite de tiempo.")
        svm_noise_results[noise_label] = {'best_params': None, 'report': "Búsqueda de hiperparámetros no completada.", 'model': None}
        continue # Skip to the next dataset

    print(f"Mejores hiperparámetros para el dataset {noise_label}: {best_svm_params}")

    # Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
    best_svm = SVC(**best_svm_params, random_state=42)

    fit_start_time = time.time()
    try:
        best_svm.fit(X_train, y_train)
        fit_time = time.time() - fit_start_time
        if fit_time > 300: # Check again for the final model training
             print(f"Entrenamiento del modelo SVM final para el dataset {noise_label} excedió el límite de tiempo ({fit_time:.2f}s). No se generará reporte.")
             svm_noise_results[noise_label] = {'best_params': best_svm_params, 'report': "Entrenamiento final excedió el límite de tiempo.", 'model': None}
             continue # Skip evaluation if final fit took too long


        # Evaluar en el conjunto de prueba
        y_test_pred = best_svm.predict(X_test)
        report = classification_report(y_test, y_test_pred)
        print(f"\nReporte de Clasificación (Conjunto de Prueba - Dataset {noise_label}):")
        print(report)
        svm_noise_results[noise_label] = {'best_params': best_svm_params, 'report': report, 'model': best_svm}

    except Exception as e:
        print(f"Error durante el entrenamiento o evaluación del modelo SVM final para el dataset {noise_label}: {e}")
        svm_noise_results[noise_label] = {'best_params': best_svm_params, 'report': f"Error: {e}", 'model': None}


print("\nResultados de SVM por nivel de ruido almacenados en el diccionario 'svm_noise_results'.")

In [ ]:
'''
Considerar clasificador MLP para los datasets construidos. Probar con diferentes
hiperparámetros,y elegir la mejor configuración de ellos, luego, mostrar el reporte
de clasificación del mejor modelo de cada dataset.
'''

# Ignorar advertencias de convergencia para simplificar la salida durante el ajuste de hiperparámetros
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Diccionario para almacenar los mejores modelos y reportes de clasificación de MLP por nivel de ruido
mlp_noise_results = {}

# Definir el rango de hiperparámetros a probar (using the same grid as in the previous MLP task)
param_grid_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'activation': ['tanh', 'relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01]
}

# Function to find the best hyperparameters using the development set with a time limit
def find_best_mlp_params_timed(X_train, y_train, X_dev, y_dev, param_grid, time_limit_per_fit=300): # time_limit_per_fit in seconds (5 minutes)
    best_score = -1
    best_params = None
    total_start_time = time.time()

    # Vamos a probar todas las combinaciones de hiperparámetros
    for hidden_layer_sizes in param_grid['hidden_layer_sizes']:
        for activation in param_grid['activation']:
            for alpha in param_grid['alpha']:
                for learning_rate_init in param_grid['learning_rate_init']:
                    # Check if overall time limit is exceeded
                    if time.time() - total_start_time > time_limit_per_fit:
                        print("Tiempo total excedido durante la búsqueda de hiperparámetros. Terminando búsqueda.")
                        return best_params # Return the best found so far

                    # Crear el clasificador con los hiperparámetros actuales
                    mlp = MLPClassifier(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation=activation,
                        solver='adam', # Fixed solver as per image
                        alpha=alpha,
                        learning_rate='constant', # Fixed learning_rate as per image
                        learning_rate_init=learning_rate_init,
                        max_iter=500, # Use a reasonable max_iter for tuning
                        random_state=42 # For reproducibility
                    )

                    # Entrenar en el conjunto de entrenamiento with time limit
                    fit_start_time = time.time()
                    try:
                        mlp.fit(X_train, y_train)
                        fit_time = time.time() - fit_start_time
                        if fit_time > time_limit_per_fit:
                            print(f"Entrenamiento de MLP con params {mlp.get_params()} excedió el límite de tiempo ({fit_time:.2f}s). Saltando a la siguiente combinación.")
                            continue # Skip evaluation if fit took too long

                        # Evaluar en el conjunto de desarrollo
                        y_dev_pred = mlp.predict(X_dev)
                        accuracy = accuracy_score(y_dev, y_dev_pred)

                        # Actualizar si encontramos un mejor modelo en el conjunto de desarrollo
                        if accuracy > best_score:
                            best_score = accuracy
                            best_params = {
                                'hidden_layer_sizes': hidden_layer_sizes,
                                'activation': activation,
                                'solver': 'adam',
                                'alpha': alpha,
                                'learning_rate': 'constant',
                                'learning_rate_init': learning_rate_init
                                }
                            print(f"Nuevo mejor score en dev: {best_score:.4f} con params: {best_params}")

                    except Exception as e:
                        print(f"Error durante el entrenamiento o evaluación de MLP con params {mlp.get_params()}: {e}")
                        continue # Continue to the next combination if there's an error


    return best_params


# Iterar sobre cada dataset dividido por nivel de ruido
for noise_label, data in split_datasets_noise.items():
    print(f"\n--- Clasificador MLP para el dataset {noise_label} ---")

    X_train, y_train = data['X_train'], data['y_train']
    X_dev, y_dev = data['X_dev'], data['y_dev']
    X_test, y_test = data['X_test'], data['y_test']

    # Encontrar los mejores hiperparámetros usando el conjunto de desarrollo con límite de tiempo
    print("Buscando los mejores hiperparámetros usando el conjunto de desarrollo (con límite de tiempo de 5 minutos por ajuste)...")
    best_mlp_params = find_best_mlp_params_timed(X_train, y_train, X_dev, y_dev, param_grid_mlp, time_limit_per_fit=300)

    if best_mlp_params is None:
        print(f"No se encontraron mejores hiperparámetros para el dataset {noise_label} dentro del límite de tiempo.")
        mlp_noise_results[noise_label] = {'best_params': None, 'report': "Búsqueda de hiperparámetros no completada.", 'model': None}
        continue # Skip to the next dataset size


    print(f"Mejores hiperparámetros para el dataset {noise_label}: {best_mlp_params}")

    # Entrenar el modelo final en el conjunto de entrenamiento con los mejores hiperparámetros encontrados en dev
    print(f"Entrenando el modelo final en el conjunto de entrenamiento con los mejores parámetros y evaluando en el conjunto de prueba...")
    # Use a higher max_iter for the final training for better convergence
    best_mlp = MLPClassifier(**best_mlp_params, max_iter=1000, random_state=42)

    fit_start_time = time.time()
    try:
        best_mlp.fit(X_train, y_train)
        fit_time = time.time() - fit_start_time
        if fit_time > 300: # Check again for the final model training
             print(f"Entrenamiento del modelo MLP final para el dataset {noise_label} excedió el límite de tiempo ({fit_time:.2f}s). No se generará reporte.")
             mlp_noise_results[noise_label] = {'best_params': best_mlp_params, 'report': "Entrenamiento final excedió el límite de tiempo.", 'model': None}
             continue # Skip evaluation if final fit took too long


        # Evaluar en el conjunto de prueba
        y_test_pred = best_mlp.predict(X_test)
        report = classification_report(y_test, y_test_pred)
        print(f"\nReporte de Clasificación (Conjunto de Prueba - Dataset {noise_label}):")
        print(report)
        mlp_noise_results[noise_label] = {'best_params': best_mlp_params, 'report': report, 'model': best_mlp}

    except Exception as e:
        print(f"Error durante el entrenamiento o evaluación del modelo MLP final para el dataset {noise_label}: {e}")
        mlp_noise_results[noise_label] = {'best_params': best_mlp_params, 'report': f"Error: {e}", 'model': None}


print("\nResultados de MLP por nivel de ruido almacenados en el diccionario 'mlp_noise_results'.")

# Restaurar advertencias
warnings.filterwarnings("default", category=ConvergenceWarning)

In [ ]:
'''
Visualización de resultados por medio de un barplot
'''

# Function to extract metrics from classification reports
def extract_metrics(results_dict):
    metrics_data = []
    for noise_level, result in results_dict.items():
        if 'report' in result and result['report'] != "Búsqueda de hiperparámetros no completada." and result['report'] != "Entrenamiento final excedió el límite de tiempo." and result['report'] is not None:
            report_lines = result['report'].split('\n')
            # Find the line with macro avg or weighted avg
            macro_avg_line = None
            weighted_avg_line = None
            for line in report_lines:
                if 'macro avg' in line:
                    macro_avg_line = line
                if 'weighted avg' in line:
                    weighted_avg_line = line

            if weighted_avg_line: # Prefer weighted avg if available
                parts = weighted_avg_line.split()
                # Extract metrics (precision, recall, f1-score, support)
                # Ensure there are enough parts before accessing
                if len(parts) >= 5:
                    metrics = {
                        'precision': float(parts[2]),
                        'recall': float(parts[3]),
                        'f1-score': float(parts[4]),
                        'support': int(parts[5])
                    }
                    metrics_data.append({'noise_level': noise_level, **metrics})
            elif macro_avg_line: # Fallback to macro avg
                 parts = macro_avg_line.split()
                 if len(parts) >= 5:
                    metrics = {
                        'precision': float(parts[2]),
                        'recall': float(parts[3]),
                        'f1-score': float(parts[4]),
                        'support': int(parts[5])
                    }
                    metrics_data.append({'noise_level': noise_level, **metrics})


    return pd.DataFrame(metrics_data)

# Extract metrics for each classifier
knn_metrics_noise = extract_metrics(knn_noise_results)
dt_metrics_noise = extract_metrics(dt_noise_results)
svm_metrics_noise = extract_metrics(svm_noise_results)
mlp_metrics_noise = extract_metrics(mlp_noise_results)

# Combine metrics into a single DataFrame for easier plotting
all_metrics_noise = pd.concat([
    knn_metrics_noise.assign(classifier='K-NN'),
    dt_metrics_noise.assign(classifier='Árbol de Decisión'),
    svm_metrics_noise.assign(classifier='SVM'),
    mlp_metrics_noise.assign(classifier='MLP')
])

# Define the order of noise levels for plotting
noise_order = ['clean', 'noisy_10', 'noisy_30']
all_metrics_noise['noise_level'] = pd.Categorical(all_metrics_noise['noise_level'], categories=noise_order, ordered=True)
all_metrics_noise = all_metrics_noise.sort_values('noise_level')


# Plotting the results
metrics_to_plot = ['precision', 'recall', 'f1-score']

for metric in metrics_to_plot:
    plt.figure(figsize=(10, 6))
    barplot = sns.barplot(x='noise_level', y=metric, hue='classifier', data=all_metrics_noise, palette='viridis')
    plt.title(f'{metric.capitalize()} across Noise Levels for Different Classifiers')
    plt.xlabel('Noise Level')
    plt.ylabel(metric.capitalize())
    plt.ylim(0, 1.1) # Set y-axis limit for better comparison
    plt.legend(title='Classifier')

    # Add text labels for the bars
    for container in barplot.containers:
        barplot.bar_label(container, fmt='%.2f', label_type='edge')

    plt.tight_layout()
    plt.savefig(os.path.join(figs, f'noise_comparison_{metric}_barplot.png')) # Save the figure
    plt.show()

print("\nBar plots generated and saved for Precision, Recall, and F1-score.")

In [ ]:
'''
Guardar en mi drive los notebooks en formato .ipynb.
'''

notebook_name = "notebook_taller4_new"

# Ruta completa de salida en Drive
ipynb_path = os.path.join(codes, f"{notebook_name}.ipynb")

# Ruta temporal para guardar el notebook actual antes de convertir
temp_ipynb_path = "/content/temp_notebook.ipynb"

# Guardar el notebook actual a una ruta temporal usando el comando mágico %notebook
# Esto asegura que nbconvert tenga un archivo .ipynb específico para trabajar
get_ipython().run_line_magic('notebook', temp_ipynb_path)

# Guardar en Drive en formato .ipynb
# Usar el archivo temporal guardado por %notebook
!jupyter nbconvert --to notebook --output "{ipynb_path}" "{temp_ipynb_path}"

print(f"Archivo guardado en Drive:\n- {ipynb_path}")